In [2]:
import pandas as pd
import numpy as np

# Load the three CSV files (generated earlier)
customers = pd.read_csv('customers.csv')
orders    = pd.read_csv('orders.csv')
order_items = pd.read_csv('order_items.csv')

# Convert date columns to datetime (for time-based operations)
orders['order_date'] = pd.to_datetime(orders['order_date'])

In [3]:
completed_orders = orders[orders['status'] == 'Completed'].copy()
completed_orders.head()

,order_id,customer_id,order_date,status,total_amount
0,1,70,2024-01-21,Completed,285
1,2,13,2024-01-24,Completed,1470
3,4,90,2024-01-08,Completed,199
4,5,52,2024-01-15,Completed,140
5,6,97,2024-01-25,Completed,199


**Q1: Total and average revenue**

In [4]:
q1 = pd.DataFrame({
    'Metric': ['Total Revenue', 'Avg Order Value', 'Total Orders', 'Unique Customers'],
    'Value': [
        round(completed_orders['total_amount'].sum(), 2),
        round(completed_orders['total_amount'].mean(), 2),
        len(completed_orders),
        completed_orders['customer_id'].nunique()
    ]
})

**Q2: Monthly revenue trend with MoM growth**

In [5]:
# Group by month (year-month)
monthly = completed_orders.groupby(
    completed_orders['order_date'].dt.to_period('M')
)['total_amount'].sum().reset_index()
monthly.columns = ['month', 'revenue']
monthly['month'] = monthly['month'].astype(str)

# Compute previous month and growth %
monthly['prev_month'] = monthly['revenue'].shift(1)
monthly['mom_growth_pct'] = round(
    (monthly['revenue'] - monthly['prev_month']) / monthly['prev_month'] * 100, 1
)

q2 = monthly[['month', 'revenue', 'prev_month', 'mom_growth_pct']]

**Q3: Revenue by customer segment**

In [6]:
# Merge completed orders with customers
segment_data = completed_orders.merge(customers, on='customer_id')

q3 = segment_data.groupby('segment').agg(
    customers=('customer_id', 'nunique'),
    orders=('order_id', 'count'),
    total_revenue=('total_amount', lambda x: round(x.sum(), 2)),
    avg_order_value=('total_amount', lambda x: round(x.mean(), 2))
).reset_index().sort_values('total_revenue', ascending=False)

**Q4: Top 10 customers by lifetime value**

In [7]:
customer_ltv = completed_orders.groupby('customer_id').agg(
    order_count=('order_id', 'count'),
    lifetime_value=('total_amount', lambda x: round(x.sum(), 2)),
    last_order_date=('order_date', 'max')
).reset_index()

# Join customer details
q4 = customer_ltv.merge(customers, on='customer_id')[['name', 'segment', 'region',
                                                     'order_count', 'lifetime_value', 'last_order_date']] \
                  .sort_values('lifetime_value', ascending=False).head(10)

**Q5: Revenue by product category**

In [8]:
# Merge order_items with completed orders
items_data = order_items.merge(completed_orders, on='order_id')

# Calculate gross revenue per item (if not already present)
items_data['gross'] = items_data['quantity'] * items_data['unit_price']

q5 = items_data.groupby('category').agg(
    units_sold=('quantity', 'sum'),
    gross_revenue=('gross', lambda x: round(x.sum(), 2)),
    avg_unit_price=('unit_price', lambda x: round(x.mean(), 2)),
    orders_containing=('order_id', 'nunique')
).reset_index().sort_values('gross_revenue', ascending=False)

**Q6: Top 5 products by revenue with share %**

In [9]:
# Total revenue from all completed orders (for share calculation)
total_revenue = (items_data['quantity'] * items_data['unit_price']).sum()

product_revenue = items_data.groupby(['product_name', 'category']).agg(
    units_sold=('quantity', 'sum'),
    revenue=('gross', lambda x: round(x.sum(), 2))
).reset_index()

product_revenue['revenue_share_pct'] = round(
    product_revenue['revenue'] / total_revenue * 100, 1
)

q6 = product_revenue.sort_values('revenue', ascending=False).head(5)

**Q7: Running revenue total by month + revenue rank**

In [10]:
# Reuse monthly DataFrame from Q2 (with month and revenue)
# Compute running total using cumsum
monthly['running_total'] = monthly['revenue'].cumsum().round(2)
monthly['revenue_rank'] = monthly['revenue'].rank(method='dense', ascending=False).astype(int)

q7 = monthly[['month', 'revenue', 'running_total', 'revenue_rank']]

**Q8: Customer percentile ranking (quartile & percentile)**

In [11]:
# Reuse customer_ltv from Q4 (has customer_id and lifetime_value)
# Assign quartiles using pd.qcut (4 equal groups)
customer_ltv['quartile'] = pd.qcut(customer_ltv['lifetime_value'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])

# Compute percentile rank (0 to 1) using rank(pct=True)
customer_ltv['percentile'] = customer_ltv['lifetime_value'].rank(pct=True).round(2)

q8 = customer_ltv[['customer_id', 'lifetime_value', 'quartile', 'percentile']] \
      .sort_values('lifetime_value', ascending=False)

### Export all sheets to Excel

In [12]:
with pd.ExcelWriter('sales_report.xlsx') as writer:
    q1.to_excel(writer, sheet_name='Revenue_KPIs', index=False)
    q2.to_excel(writer, sheet_name='Monthly_Trend', index=False)
    q3.to_excel(writer, sheet_name='By_Segment', index=False)
    q4.to_excel(writer, sheet_name='Top_Customers', index=False)
    q5.to_excel(writer, sheet_name='By_Category', index=False)
    q6.to_excel(writer, sheet_name='Top_Products', index=False)
    q7.to_excel(writer, sheet_name='Running_Total', index=False)
    q8.to_excel(writer, sheet_name='Customer_Percentiles', index=False)

print("Excel report generated successfully!")

Excel report generated successfully!
